# Evaluation

This notebook contains the text quality evaluation of the augmented text data for the Kaggle MBTI dataset. 
The results of backtranslation, syonym swapping, random swapping and random deletion are tested using:

- BLEU Score   
- self-BLEU
- TTR

**BLEU** - Compares predicitons with ground truth, measures how close to ground truth prediceted sentences are.

**self-BLEU** - Measures how diverse predicted sentences are among each other.

**TTR** - Calculates ratio of unique words to all words.

In [1]:
import pandas as pd
import evaluate
from nltk.translate.bleu_score import sentence_bleu

In [2]:
# reading data
bt_df = pd.read_csv("../data/mbti_augmented_agg/bt_augmented_agg.csv", sep = "\t", quoting = 1)
syn_df = pd.read_csv("../data/mbti_augmented_agg/syn_augmented_agg.csv", sep = "\t", quoting = 1)
sw_df = pd.read_csv("../data/mbti_augmented_agg/sw_augmented_agg.csv", sep = "\t", quoting = 1)
del_df = pd.read_csv("../data/mbti_augmented_agg/del_augmented_agg.csv", sep = "\t", quoting = 1)


In [ ]:
bleu = evaluate.load("bleu")

# bleu needs references in nested list

#BT
predictions_bt = bt_df["post_augmented"].tolist()
references_bt = [[ref] for ref in bt_df["post"].tolist()]

results_bt = bleu.compute(predictions=predictions_bt, references=references_bt)

# SYN
predictions_syn = syn_df["post_augmented"].tolist()
references_syn = [[ref] for ref in syn_df["post"].tolist()]

results_syn = bleu.compute(predictions=predictions_syn, references=references_syn)

#RSW
predictions_sw = sw_df["post_augmented"].tolist()
references_sw = [[ref] for ref in sw_df["post"].tolist()]

results_sw = bleu.compute(predictions=predictions_sw, references=references_sw)

#RDEL
predictions_del = del_df["post_augmented"].tolist()
references_del = [[ref] for ref in del_df["post"].tolist()]

results_del = bleu.compute(predictions=predictions_del, references=references_del)

# print results
print(f"BLEU-Score: \nBacktranslation: {results_bt} \nSynonym-Swapping: {results_syn} \nRandom Swapping: {results_sw} \nRandom Deletion: {results_del}")

BLEU-Score: 
Backtranslation: {'bleu': 0.539692488067903, 'precisions': [0.8144192555356559, 0.6208661845270877, 0.49199313541562933, 0.39133311557547135], 'brevity_penalty': 0.9661809098211563, 'length_ratio': 0.9667400948746458, 'translation_length': 2768326, 'reference_length': 2863568} 
Synonym-Swapping: {'bleu': 0.6581311597452123, 'precisions': [0.8178212996492182, 0.7036013787726741, 0.6118863467238588, 0.5328364809412314], 'brevity_penalty': 1.0, 'length_ratio': 1.064334250521422, 'translation_length': 882315, 'reference_length': 828983} 
Random Swapping: {'bleu': 0.48728096849212676, 'precisions': [0.9170944040740027, 0.5181570963029554, 0.3891209994726808, 0.30490014868107684], 'brevity_penalty': 1.0, 'length_ratio': 1.055716990061524, 'translation_length': 223073, 'reference_length': 211300} 
Random Deletion: {'bleu': 0.5226308556976901, 'precisions': [0.927542729055758, 0.7167300166434258, 0.5572819696918407, 0.43596159507257537], 'brevity_penalty': 0.8244073606262522, 'len

In [ ]:

def self_bleu(sentences):
    scores = []
    for i, hypothesis in enumerate(sentences):
        references = [s.split() for j, s in enumerate(sentences) if j != i] # I am comparing each sentence with the whole sample, except for itself; split because NLTK needs word tokenisation
        score = sentence_bleu(references, hypothesis.split()) # split() again
        scores.append(score)
    return sum(scores) / len(scores)



import random
sample_bt = random.sample(predictions_bt, 500) #sample size of 500 for computational efficiency
sample_syn = random.sample(predictions_syn, 500)
sample_sw = random.sample(predictions_sw, 500)
sample_del = random.sample(predictions_del, 500)

print(f"Self-BLEU Score: \nBacktranslation: {self_bleu(sample_bt)}\nSynonym-Swapping: {self_bleu(sample_syn)} \nRandom Swapping: {self_bleu(sample_sw)} \nRandom Deletion: {self_bleu(sample_del)}")



c:\Users\Tim\src\MA\.venv\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
c:\Users\Tim\src\MA\.venv\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
c:\Users\Tim\src\MA\.venv\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  war

Self-BLEU Score: 
Backtranslation: 0.3940926589258324
Synonym-Swapping: 0.3598384426946354 
Random Swapping: 0.24936971671645544 
Random Deletion: 0.2899214907712916


In [ ]:
# TTR
# number unique words (types) / number of all words (tokens)

def calculate_ttr(df):
    all_tokens = " ".join(df.astype(str)).split() # putting all words into one big string, pandas into list
    unique_words = set(all_tokens) # filtering unique words, list into set. set can't hold duplicates
    ttr = len(unique_words) / len(all_tokens)
    return ttr

ttr_original_bt = round(calculate_ttr(bt_df["post"]),3)
ttr_augmented_bt = round(calculate_ttr(bt_df["post_augmented"]),3)

ttr_original_syn = round(calculate_ttr(syn_df["post"]),3)
ttr_augmented_syn = round(calculate_ttr(syn_df["post_augmented"]),3)

ttr_original_sw = round(calculate_ttr(sw_df["post"]),3)
ttr_augmented_sw = round(calculate_ttr(sw_df["post_augmented"]),3)

ttr_original_del = round(calculate_ttr(del_df["post"]),3)
ttr_augmented_del = round(calculate_ttr(del_df["post_augmented"]),3)

print(f"TTR: BT Original: {ttr_original_bt}, BT Augmented: {ttr_augmented_bt}, Difference: {round(ttr_augmented_bt-ttr_original_bt,3)}")
print(f"TTR: SYN Original: {ttr_original_syn}, SYN Augmented: {ttr_augmented_syn}, Difference: {round(ttr_augmented_syn-ttr_original_syn,3)}")
print(f"TTR: RSW Original: {ttr_original_sw}, RSW Augmented: {ttr_augmented_sw}, Difference: {round(ttr_augmented_sw-ttr_original_sw,3)}")
print(f"TTR: RDEL Original: {ttr_original_del}, RDEL Augmented: {ttr_augmented_del}, Difference: {round(ttr_augmented_del-ttr_original_del,3)}")

TTR: BT Original: 0.027, BT Augmented: 0.03, Difference: 0.003
TTR: SYN Original: 0.059, SYN Augmented: 0.046, Difference: -0.013
TTR: RSW Original: 0.115, RSW Augmented: 0.097, Difference: -0.018
TTR: RDEL Original: 0.115, RDEL Augmented: 0.108, Difference: -0.007
